# PySpark WordCount Example with Hadoop HDFS

This notebook demonstrates a classic MapReduce WordCount problem using PySpark on the Hadoop cluster.

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split, lower, col
from pyspark.sql.types import StructType, StructField, StringType
import re

In [ ]:
# Create SparkSession
spark = SparkSession.builder \
    .appName("WordCount-Example") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://hadoop-master:9000") \
    .getOrCreate()

print("Spark Session created!")

In [ ]:
# Create sample text data and save to HDFS
sample_text = """Apache Spark is a powerful open source big data processing engine
Spark can run on Hadoop YARN and Spark supports multiple programming languages
PySpark is the Python API for Apache Spark
Spark is built on top of HDFS and MapReduce framework
Apache Spark provides fast and unified computing for big data processing
We can use Spark for data processing machine learning and streaming
Spark SQL provides SQL interface for Spark DataFrames
Spark Streaming allows real time data processing
MLlib is the machine learning library for Apache Spark
GraphX is for graph processing in Apache Spark"""

# Save to HDFS
input_path = "hdfs://hadoop-master:9000/user/jupyter/wordcount_input.txt"

try:
    # Create RDD and save to HDFS
    lines = spark.sparkContext.parallelize([sample_text])
    lines.saveAsTextFile(input_path.replace(".txt", ""))
    print(f"Sample text saved to HDFS: {input_path}")
except Exception as e:
    print(f"Error saving to HDFS: {e}")

In [ ]:
# Read from HDFS and perform WordCount
try:
    # Read text file from HDFS
    text_path = "hdfs://hadoop-master:9000/user/jupyter/wordcount_input"
    text_rdd = spark.sparkContext.textFile(text_path)
    
    print(f"Read text from HDFS: {text_path}")
    print(f"Number of lines: {text_rdd.count()}")
    
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Method 1: Using RDD (MapReduce style)
print("\n=== Method 1: RDD-based WordCount ===")

def clean_word(word):
    # Remove punctuation and convert to lowercase
    return re.sub(r'[^a-zA-Z0-9]', '', word).lower()

# Map and Reduce
word_counts_rdd = text_rdd \
    .flatMap(lambda line: line.split()) \
    .map(lambda word: (clean_word(word), 1)) \
    .filter(lambda x: len(x[0]) > 0) \
    .reduceByKey(lambda a, b: a + b) \
    .sortBy(lambda x: x[1], ascending=False)

# Get results
results_rdd = word_counts_rdd.take(20)
print("\nTop 20 Words (using RDD):")
for word, count in results_rdd:
    print(f"{word}: {count}")

In [ ]:
# Method 2: Using DataFrame and SQL (modern Spark style)
print("\n=== Method 2: DataFrame-based WordCount ===")

from pyspark.sql.functions import regexp_replace, trim

# Read as DataFrame
text_df = spark.read.text(text_path)

# Perform WordCount using DataFrame operations
word_counts_df = text_df \
    .select(explode(split(lower(col("value")), " ")).alias("word")) \
    .select(regexp_replace(col("word"), "[^a-z0-9]", "").alias("word")) \
    .filter(col("word") != "") \
    .groupBy("word") \
    .count() \
    .sort(col("count").desc())

print("\nTop 20 Words (using DataFrame):")
word_counts_df.show(20, truncate=False)

In [ ]:
# Save results to HDFS
output_path = "hdfs://hadoop-master:9000/user/jupyter/wordcount_output"

try:
    word_counts_df.write.mode("overwrite").csv(output_path)
    print(f"\nResults saved to HDFS: {output_path}")
except Exception as e:
    print(f"Error saving results: {e}")

In [ ]:
# Verify saved results
try:
    results_df = spark.read.csv(output_path, header=False)
    print("\nSaved results from HDFS:")
    results_df.show(20, truncate=False)
except Exception as e:
    print(f"Error reading results: {e}")

In [ ]:
# Get statistics
print("\n=== Word Count Statistics ===")
total_words = word_counts_df.agg({"count": "sum"}).collect()[0][0]
unique_words = word_counts_df.count()
max_count = word_counts_df.agg({"count": "max"}).collect()[0][0]
min_count = word_counts_df.agg({"count": "min"}).collect()[0][0]
avg_count = word_counts_df.agg({"count": "avg"}).collect()[0][0]

print(f"Total words: {total_words}")
print(f"Unique words: {unique_words}")
print(f"Max count: {max_count}")
print(f"Min count: {min_count}")
print(f"Average count: {avg_count:.2f}")

In [ ]:
# Visualization (if matplotlib is available)
import matplotlib.pyplot as plt

# Get top 10 words
top_10 = word_counts_df.limit(10).collect()

words = [row[0] for row in top_10]
counts = [row[1] for row in top_10]

plt.figure(figsize=(10, 6))
plt.bar(words, counts, color='steelblue')
plt.xlabel('Words')
plt.ylabel('Count')
plt.title('Top 10 Most Frequent Words')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nChart displayed!")

In [ ]:
# Stop Spark Session
spark.stop()
print("WordCount example completed!")